In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & INSTALAÇÕES
# Instala e importa dependências para execução no Kaggle (GPU T4 x2)
# =============================================================================

import subprocess, sys, os, math, warnings, unicodedata, heapq
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

def pip_install(package):
    """Instala um pacote via pip de forma silenciosa."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

print("\u{1F4E6} Instalando dependências...")
pip_install("gudhi")       # Persistent homology, diagramas de persistência
pip_install("giotto-tda")  # Pipeline TDA (Topological Data Analysis)
pip_install("keras>=3.0")  # Keras 3 multi-backend
pip_install("keras-nlp")   # KerasNLP — suporte a Gemma 2
pip_install("networkx")    # Grafo de dependências (Algoritmo JP)
pip_install("scipy")       # Transporte ótimo
print("\u2705 Dependências instaladas!")

# Configura backend Keras: JAX (melhor desempenho em GPU T4)
os.environ["KERAS_BACKEND"] = "jax"

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
print(f"\u{1F527} Python {sys.version.split()[0]} | NumPy {np.__version__}")
print(f"   Keras backend: {os.environ.get('KERAS_BACKEND', 'tensorflow')}")

# 🧠 AFASIA — Inteligência Artificial Pictórica (IAP)
## Kaggle Notebook — Gemma 4 Good Hackathon 2025

**Projeto:** Atlas Topológico da Afasia + Algoritmo JP  
**Autor:** João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)  
**Competição:** [Gemma 4 Good Hackathon — Kaggle / Google DeepMind](https://www.kaggle.com/competitions/gemma-4-good)  
**Repositório:** [github.com/joaopedropassostocantins/AFASIA](https://github.com/joaopedropassostocantins/AFASIA)

---

### Inteligência Artificial Pictórica (IAP) — Teoria e Motivação

**"Construir aviões em vez de imitar pássaros."**

A IA convencional imita a linguagem humana — tokeniza texto, aprende distribuições estatísticas e gera respostas probabilísticas. A **IAP** propõe algo fundamentalmente diferente: operar no nível **pré-linguístico** do pensamento, onde o significado existe como **estrutura topológica** antes de qualquer palavra.

### Equação Central: $G \approx I + F$

$$G \approx I + F$$

| Variável | Nome | Descrição |
|----------|------|-----------|
| $G$ | **Dinâmica do Conhecimento** | O estado em transformação — o pensamento em movimento |
| $I$ | **Incerteza** | O espaço topológico atual, incompleto, do usuário |
| $F$ | **Flexibilidade** | A capacidade de encontrar o próximo passo ótimo |

### Aplicação: Afasia

A **afasia** é uma síndrome neurológica adquirida (AVC, trauma) que compromete o acesso ao código linguístico. Pacientes afásicos pensam em conceitos antes de conseguir verbalizá-los — operam no espaço topológico pré-linguístico que a IAP modela.

### Pipeline deste Notebook

```
Pictograma ARASAAC (palavra em pt-BR)
    ↓  [Gemma 2 — hidden state da última camada, token BOS]
Embedding semântico ℝⁿ → estado IAP 5D
    ↓  [Motor Topológico: Diagrama de Persistência + Wasserstein]
Matriz de distâncias topológicas N×N
    ↓  [Algoritmo JP: Dijkstra regressivo com waypoints]
Plano de comunicação: "Fome → Comer → Maçã"
```

### 🎥 Demonstração da Interface Web

<div style="position:relative;padding-bottom:56.25%;height:0;overflow:hidden;max-width:640px;background:#1e293b;border-radius:8px;">
  <iframe
    src="https://www.youtube.com/embed/SEU_VIDEO_AQUI"
    title="AFASIA IAP — Demonstração"
    frameborder="0"
    allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
    allowfullscreen
    style="position:absolute;top:0;left:0;width:100%;height:100%;">
  </iframe>
</div>

*Substitua `SEU_VIDEO_AQUI` pelo ID do vídeo no YouTube após gravar a demonstração.*

In [ ]:
# =============================================================================
# CÉLULA 3 — GEMMA 2 COMO EXTRATOR DE EMBEDDINGS
#
# O Gemma 2 é usado como extrator semântico — capturamos o hidden state
# da ÚLTIMA CAMADA no token BOS (posição 0, equivalente ao [CLS] do BERT)
# para cada rótulo de pictograma em português, gerando o espaço vetorial
# que alimenta o Motor Topológico IAP.
#
# Modo Gemma real:    carrega gemma2_2b_en, float16, extrai cls_hidden
# Modo simulado:      embedding determinístico por hash sem dependências extras
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

import keras_nlp
import keras

GEMMA_LOADED     = False
_gemma_backbone  = None
_gemma_tokenizer = None

print("\u{1F916} Inicializando Gemma 2 (gemma2_2b_en)...")
print("   Modo: extrator de hidden state da última camada (token BOS)")

try:
    _gemma_backbone = keras_nlp.models.GemmaBackbone.from_preset(
        "gemma2_2b_en",
        dtype="float16"   # float16 para economia de VRAM na T4
    )
    _gemma_tokenizer = keras_nlp.models.GemmaTokenizer.from_preset("gemma2_2b_en")
    GEMMA_LOADED = True
    print(f"\u2705 Gemma 2 carregado! Parâmetros: {_gemma_backbone.count_params():,}")
except Exception as e:
    print(f"\u26A0\uFE0F  Gemma 2 indisponível ({type(e).__name__}). Modo simulado ativado.")
    print("   No Kaggle com token configurado, o modelo real será usado.")


# ─── Dicionários IAP — PORTA FIEL DO BACKEND TYPESCRIPT ──────────────────────
# Fonte: artifacts/api-server/src/routes/iap/index.ts
# CAT_STATE_VECTORS: [urgência_vital, valência_emocional, concretude_espacial,
#                     relacionalidade, agentividade]

CAT_STATE_VECTORS: Dict[str, List[float]] = {
    'necessidades': [9, 2, 1, 2, 1],
    'sentimentos':  [2, 9, 2, 1, 2],
    'lugares':      [1, 2, 9, 2, 1],
    'pessoas':      [2, 1, 2, 9, 2],
    'acoes':        [1, 2, 1, 2, 9],
    'outros':       [5, 5, 5, 5, 5],
}

# Porta fiel de ATLAS_CAT_KEYWORDS do TypeScript (sem modificações semânticas)
ATLAS_CAT_KEYWORDS: Dict[str, List[str]] = {
    'necessidades': ['agua', 'comida', 'comer', 'beber', 'banheiro', 'toalete',
                     'remedio', 'medicina', 'ajuda', 'socorro', 'dormir', 'descanso',
                     'frio', 'calor', 'fome', 'sede', 'higiene', 'banho', 'dente',
                     'roupa', 'sapato'],
    'sentimentos':  ['feliz', 'alegre', 'triste', 'dor', 'doer', 'medo', 'ansioso',
                     'ansiedade', 'cansado', 'irritado', 'confuso', 'nervoso', 'raiva',
                     'amor', 'gostar', 'emocao', 'choro', 'chorar', 'rir', 'saudade',
                     'angustia', 'stress', 'depressao'],
    'lugares':      ['casa', 'hospital', 'escola', 'trabalho', 'parque', 'quarto',
                     'rua', 'jardim', 'cidade', 'praia', 'mercado', 'supermercado',
                     'farmacia', 'clinica', 'banheiro'],
    'pessoas':      ['eu', 'mae', 'pai', 'familia', 'medico', 'enfermeiro', 'amigo',
                     'cuidador', 'filho', 'irmao', 'avo', 'pessoa', 'homem', 'mulher',
                     'crianca'],
    'acoes':        ['quero', 'nao', 'sim', 'parar', 'ir', 'vir', 'dar', 'chamar',
                     'ajudar', 'fazer', 'ver', 'ouvir', 'falar', 'andar', 'correr',
                     'sentar', 'levantar', 'brincar', 'trabalhar', 'estudar', 'dormir'],
}


def _normalize_pt(s: str) -> str:
    """Normaliza string PT-BR: minúsculas, remove diacríticos (igual ao TypeScript)."""
    nfd = unicodedata.normalize('NFD', s.lower())
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')


def inferir_categoria(palavra: str) -> str:
    """
    Infere a categoria semântica IAP de um pictograma pelo rótulo.
    Porta fiel de inferCategoria() do backend TypeScript.

    Normaliza (lowercase + remove acentos) e verifica pertencimento
    bidirecional: keyword-in-lower OU lower-in-keyword.
    """
    lower = _normalize_pt(palavra)
    for cat, keywords in ATLAS_CAT_KEYWORDS.items():
        norm_kws = [_normalize_pt(k) for k in keywords]
        if any(k in lower or lower in k for k in norm_kws):
            return cat
    return 'outros'


def extrair_embedding_gemma(palavra: str, dimensao: int = 5) -> np.ndarray:
    """
    Extrai embedding semântico via Gemma 2 e projeta para o espaço IAP 5D.

    Modo Gemma real:
        Tokeniza a palavra, executa forward pass pelo GemmaBackbone.
        Extrai o hidden state da última camada no token BOS (posição 0),
        que acumula contexto global (equivalente ao [CLS] do BERT).
        Projeta para `dimensao` via média de blocos contíguos.

    Modo simulado (sem Gemma disponível):
        Usa CAT_STATE_VECTORS da categoria inferida + ruído determinístico
        por hash da palavra — garante reprodutibilidade sem dependências.

    Normalização final: escala para [1, 9] — compatível com CAT_STATE_VECTORS.

    Args:
        palavra:  Rótulo do pictograma em PT-BR
        dimensao: Dimensionalidade do vetor de saída (default: 5 = espaço IAP)

    Returns:
        np.ndarray (dimensao,) normalizado em [1, 9]
    """
    if GEMMA_LOADED and _gemma_backbone is not None and _gemma_tokenizer is not None:
        # ── Modo Gemma 2 real ────────────────────────────────────────────────
        token_ids    = _gemma_tokenizer([palavra])          # shape: (1, seq_len)
        padding_mask = (np.array(token_ids) != 0)           # shape: (1, seq_len)

        # Forward pass — output: tensor (1, seq_len, hidden_dim)
        hidden_states = _gemma_backbone(
            {"token_ids": token_ids, "padding_mask": padding_mask},
            training=False
        )

        # CLS representation: token BOS (posição 0), última camada do backbone
        cls_hidden = np.array(hidden_states[0, 0, :], dtype=np.float32)  # (hidden_dim,)

        # Projeta para `dimensao` via média de blocos contíguos
        hidden_dim = cls_hidden.shape[0]
        block_size = max(1, hidden_dim // dimensao)
        raw        = cls_hidden[:block_size * dimensao].reshape(dimensao, block_size).mean(axis=1)

    else:
        # ── Modo simulado determinístico ─────────────────────────────────────
        cat  = inferir_categoria(palavra)
        base = np.array(CAT_STATE_VECTORS.get(cat, CAT_STATE_VECTORS['outros']),
                        dtype=np.float32)
        word_hash = sum(ord(c) * (i + 1) for i, c in enumerate(palavra)) % 1000
        rng  = np.random.default_rng(seed=word_hash)
        raw  = base + rng.normal(0, 0.3, dimensao).astype(np.float32)

    # Normaliza para [1, 9] — compatível com CAT_STATE_VECTORS IAP
    v_min, v_max = raw.min(), raw.max()
    span  = max(float(v_max - v_min), 1e-6)
    estado = 1.0 + 8.0 * (raw - v_min) / span
    return estado.astype(np.float32)


# ─── Teste de sanidade ───────────────────────────────────────────────────────
print()
print("\u{1F9EA} Teste extrair_embedding_gemma() + inferir_categoria():")
for palavra in ['água', 'dor', 'família', 'casa', 'quero', 'maçã', 'fome', 'comer']:
    estado = extrair_embedding_gemma(palavra)
    cat    = inferir_categoria(palavra)
    print(f"   '{palavra}' [{cat:12s}] → estado={np.round(estado, 2)}")
print()
print("\u2705 Extrator de embeddings Gemma 2 pronto!")

In [ ]:
# =============================================================================
# CÉLULA 4 — MOTOR TOPOLÓGICO IAP
#
# Porta fiel do backend TypeScript (artifacts/api-server/src/routes/iap/index.ts):
#   generateSimulatedPersistenceDiagram → gerar_diagrama_persistencia()
#   computeWasserstein                  → calcular_wasserstein()
#   pairwiseTopo                        → distancias_pairwise()
#   classicalMDS                        → mds_classico()
#
# INTEGRAÇÃO GEMMA:
#   distancias_pairwise() chama extrair_embedding_gemma() para cada pictograma.
#   O embedding 5D alimenta gerar_diagrama_persistencia() diretamente.
#   Pipeline: Gemma 2 → estado 5D → Diagrama → Wasserstein → MDS
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

@dataclass
class PontoPersistencia:
    """
    Ponto em um Diagrama de Persistência.

    birth:    filtração em que a feature topológica nasce
    death:    filtração em que a feature morre
    dimensao: 0=componentes conectadas(H0), 1=buracos(H1)
    lifetime: death - birth
    """
    birth:    float
    death:    float
    dimensao: int
    lifetime: float


def gerar_diagrama_persistencia(
    estado: np.ndarray,
    seed: int
) -> List[PontoPersistencia]:
    """
    Gera Diagrama de Persistência a partir de um vetor de estado IAP.

    O vetor de estado (saída de extrair_embedding_gemma) codifica propriedades
    pré-linguísticas de um conceito. O diagrama captura:
      H0 (dim=0): componentes conectadas — estrutura de clusters
      H1 (dim=1): buracos/loops — complexidade cíclica

    Porta fiel de generateSimulatedPersistenceDiagram() do TypeScript.
    """
    estado_list = list(estado)
    n           = len(estado_list)
    media       = sum(estado_list) / n
    variancia   = sum((v - media) ** 2 for v in estado_list) / n
    pontos: List[PontoPersistencia] = []

    # H0: componentes conectadas (número cresce com intensidade do estado)
    num_h0 = max(2, int(media / 3) + 1)
    for i in range(num_h0):
        birth = (i * 0.8 + seed * 0.1) % 2.0
        death = birth + 0.3 + estado_list[i % n] * 0.15
        pontos.append(PontoPersistencia(
            birth=round(birth, 2), death=round(death, 2),
            dimensao=0, lifetime=round(death - birth, 2)
        ))

    # H1: buracos topológicos (número cresce com variância / diversidade)
    num_h1 = max(1, int(variancia / 5) + 1)
    for i in range(num_h1):
        birth = 0.5 + (i * 0.7 + seed * 0.2) % 1.5
        death = birth + 0.2 + estado_list[(i + 1) % n] * 0.1
        pontos.append(PontoPersistencia(
            birth=round(birth, 2), death=round(death, 2),
            dimensao=1, lifetime=round(death - birth, 2)
        ))

    return pontos


def calcular_wasserstein(
    diag1: List[PontoPersistencia],
    diag2: List[PontoPersistencia]
) -> float:
    """
    Distância de Wasserstein entre dois Diagramas de Persistência.

    Transporte ótimo sobre H0 (dim=0):
    - Pares alinhados:    distância Euclidiana entre os pontos
    - Pontos sem par:     projeção na diagonal = lifetime / √2

    Porta fiel de computeWasserstein() do TypeScript.
    """
    h0_1 = [p for p in diag1 if p.dimensao == 0]
    h0_2 = [p for p in diag2 if p.dimensao == 0]
    total = 0.0
    n     = max(len(h0_1), len(h0_2))

    for i in range(n):
        b1 = h0_1[i] if i < len(h0_1) else None
        b2 = h0_2[i] if i < len(h0_2) else None
        if b1 is not None and b2 is not None:
            total += math.sqrt((b1.birth - b2.birth) ** 2 + (b1.death - b2.death) ** 2)
        elif b1 is not None:
            total += b1.lifetime / math.sqrt(2)
        elif b2 is not None:
            total += b2.lifetime / math.sqrt(2)

    return round(total, 4)


def distancias_pairwise(
    pictos: List[Dict],
    wass_scale: float = 3.0
) -> np.ndarray:
    """
    Matriz de Distâncias de Wasserstein para todos os pares de pictogramas.

    INTEGRAÇÃO GEMMA: chama extrair_embedding_gemma() para cada pictograma,
    obtendo o estado 5D que alimenta gerar_diagrama_persistencia().
    Ruído determinístico (noise = (id%7)/10 * sin(idx*(id%13))) preserva
    a variabilidade intra-categoria exatamente como no TypeScript.

    Porta fiel de pairwiseTopo() do TypeScript.

    Returns:
        np.ndarray (N, N) — matriz simétrica, valores em [0, 1]
    """
    n = len(pictos)

    # Extrai estado via Gemma (ou simulado) + ruído determinístico por ID
    state_vectors = []
    for p in pictos:
        base    = extrair_embedding_gemma(p['palavra'], dimensao=5)
        noise_f = (p['id'] % 7) / 10.0
        sv = np.array([
            max(1.0, float(base[idx]) + noise_f * math.sin(idx * (p['id'] % 13)))
            for idx in range(5)
        ], dtype=np.float32)
        state_vectors.append(sv)

    # Gera diagramas de persistência (seed = id % 100, igual ao TypeScript)
    diagramas = [
        gerar_diagrama_persistencia(sv, seed=pictos[i]['id'] % 100)
        for i, sv in enumerate(state_vectors)
    ]

    # Wasserstein pairwise com normalização
    dist = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i + 1, n):
            w = calcular_wasserstein(diagramas[i], diagramas[j])
            normalized = min(1.0, w / wass_scale)
            dist[i, j] = normalized
            dist[j, i] = normalized

    return dist


def mds_classico(dist: np.ndarray) -> np.ndarray:
    """
    Projeção 2D por MDS Clássico (Multi-Dimensional Scaling).

    Projeta N pictogramas do espaço de Wasserstein para ℝ² via iteração
    de potência sobre os dois maiores autovetores da matriz B de dupla
    centração. Porta fiel de classicalMDS() do TypeScript.

    Returns:
        np.ndarray (N, 2) — coordenadas (x, y) do Atlas Topológico
    """
    n = dist.shape[0]
    if n == 0: return np.zeros((0, 2))
    if n == 1: return np.zeros((1, 2))
    if n == 2: return np.array([[-dist[0, 1] / 2, 0.0], [dist[0, 1] / 2, 0.0]])

    D2         = dist ** 2
    row_means  = D2.mean(axis=1)
    col_means  = D2.mean(axis=0)
    grand_mean = D2.mean()
    B = -0.5 * (D2 - row_means[:, None] - col_means[None, :] + grand_mean)

    def power_iterate(M: np.ndarray, seed_val: float) -> Tuple[np.ndarray, float]:
        v  = np.array([math.sin(i * 1.618 + seed_val) for i in range(n)])
        nm = np.linalg.norm(v)
        v  = v / nm if nm > 1e-12 else np.ones(n) / math.sqrt(n)
        for _ in range(150):
            Mv = M @ v
            nm = np.linalg.norm(Mv)
            v  = Mv / nm if nm > 1e-12 else v
        return v, float(v @ (M @ v))

    v1, lam1 = power_iterate(B, 0.0)
    B2        = B - lam1 * np.outer(v1, v1)
    v2, lam2  = power_iterate(B2, 1.5)
    s1 = math.sqrt(max(0.0, lam1))
    s2 = math.sqrt(max(0.0, lam2))

    return np.column_stack([np.round(v1 * s1, 3), np.round(v2 * s2, 3)])


# ─── Teste do Motor Topológico ───────────────────────────────────────────────
print("\u{1F52C} Teste do Motor Topológico IAP (Gemma 2 → Wasserstein):")
print()
for par in [('fome', 'comer'), ('comer', 'maçã'), ('fome', 'maçã')]:
    e1 = extrair_embedding_gemma(par[0])
    e2 = extrair_embedding_gemma(par[1])
    d1 = gerar_diagrama_persistencia(e1, seed=1)
    d2 = gerar_diagrama_persistencia(e2, seed=42)
    w  = calcular_wasserstein(d1, d2)
    print(f"   d_Wasserstein({par[0]!r:8s}, {par[1]!r:8s}) = {w:.4f}")
print()
print("\u2705 Motor Topológico funcionando com embeddings Gemma 2!")

In [ ]:
# =============================================================================
# CÉLULAS 5 & 6 — ALGORITMO JP + SIMULAÇÃO "FOME → COMER → MAÇÃ"
#
# Algoritmo JP: porta fiel de runJpAlgorithmLogic() do backend TypeScript.
# Dijkstra regressivo sobre grafo completo de Wasserstein.
#
# Simulação com waypoints explícitos:
#   Passo 1: planejar('fome' → 'comer')  — necessidade → ação
#   Passo 2: planejar('comer' → 'maçã')  — ação → objetivo
# O caminho composto garante que TODOS os 3 pictogramas aparecem.
#
# G ≈ I + F:
#   G = caminho Fome → Comer → Maçã (dinâmica)
#   I = pictograma 'fome' selecionado (incerteza atual)
#   F = 3 vizinhos topológicos de 'fome' (flexibilidade)
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

# ─── Tipos de dados ──────────────────────────────────────────────────────────

@dataclass
class PassoPlano:
    """Um passo no plano de comunicação gerado pelo Algoritmo JP."""
    numero_passo:          int
    de_pictograma:         str
    para_pictograma:       str
    distancia_wasserstein: float
    distancia_topologica:  float


@dataclass
class ResultadoJP:
    """Resultado completo do planejamento regressivo."""
    sucesso:            bool
    caminho:            List[PassoPlano]
    custo_total:        float
    iteracoes:          int
    mensagem:           str
    vizinhos_imediatos: List[Dict]


# ─── Conjunto de pictogramas IAP ─────────────────────────────────────────────
PICTOGRAMAS_IAP = [
    # Necessidades
    {'id':  1, 'palavra': 'água',     'categoria': inferir_categoria('água')},
    {'id':  2, 'palavra': 'comida',   'categoria': inferir_categoria('comida')},
    {'id':  3, 'palavra': 'fome',     'categoria': inferir_categoria('fome')},
    {'id':  4, 'palavra': 'sede',     'categoria': inferir_categoria('sede')},
    {'id':  5, 'palavra': 'dor',      'categoria': inferir_categoria('dor')},
    {'id':  6, 'palavra': 'remédio',  'categoria': inferir_categoria('remédio')},
    {'id':  7, 'palavra': 'ajuda',    'categoria': inferir_categoria('ajuda')},
    {'id':  8, 'palavra': 'banheiro', 'categoria': inferir_categoria('banheiro')},
    # Sentimentos
    {'id': 11, 'palavra': 'feliz',    'categoria': inferir_categoria('feliz')},
    {'id': 12, 'palavra': 'triste',   'categoria': inferir_categoria('triste')},
    {'id': 13, 'palavra': 'medo',     'categoria': inferir_categoria('medo')},
    {'id': 14, 'palavra': 'cansado',  'categoria': inferir_categoria('cansado')},
    # Lugares
    {'id': 21, 'palavra': 'casa',     'categoria': inferir_categoria('casa')},
    {'id': 22, 'palavra': 'hospital', 'categoria': inferir_categoria('hospital')},
    {'id': 23, 'palavra': 'escola',   'categoria': inferir_categoria('escola')},
    # Pessoas
    {'id': 31, 'palavra': 'eu',       'categoria': inferir_categoria('eu')},
    {'id': 32, 'palavra': 'família',  'categoria': inferir_categoria('família')},
    {'id': 33, 'palavra': 'médico',   'categoria': inferir_categoria('médico')},
    {'id': 34, 'palavra': 'amigo',    'categoria': inferir_categoria('amigo')},
    # Ações
    {'id': 41, 'palavra': 'quero',    'categoria': inferir_categoria('quero')},
    {'id': 42, 'palavra': 'sim',      'categoria': inferir_categoria('sim')},
    {'id': 43, 'palavra': 'não',      'categoria': inferir_categoria('não')},
    {'id': 44, 'palavra': 'comer',    'categoria': inferir_categoria('comer')},
    {'id': 45, 'palavra': 'ir',       'categoria': inferir_categoria('ir')},
    {'id': 46, 'palavra': 'maçã',     'categoria': inferir_categoria('maçã')},
]


# ─── Motor Dijkstra do Algoritmo JP ─────────────────────────────────────────

class AlgoritmoJP:
    """
    Motor de Planejamento Regressivo da IAP.

    Cada aresta do grafo completo tem peso = Distância de Wasserstein entre
    os Diagramas de Persistência, gerados a partir dos embeddings Gemma 2.
    """

    def __init__(self, pictogramas: List[Dict]):
        self.pictogramas     = pictogramas
        self.idx_por_id      = {p['id']: i for i, p in enumerate(pictogramas)}
        self.idx_por_palavra = {
            _normalize_pt(p['palavra']): i for i, p in enumerate(pictogramas)
        }
        print(f"   \u{1F517} Grafo JP: {len(pictogramas)} nós")
        print("   \u{1F916} Computando Gemma 2 \u2192 embeddings \u2192 Wasserstein pairwise...")
        self.dist_matrix = distancias_pairwise(pictogramas)
        print(f"   \u{1F4D0} Matriz {self.dist_matrix.shape} calculada")
        self.coords = mds_classico(self.dist_matrix)
        print(f"   \u{1F5FA}\uFE0F  Atlas 2D via MDS Clássico pronto")

    def _resolver(self, id_ou_palavra: str) -> Optional[int]:
        try:
            return self.idx_por_id.get(int(id_ou_palavra))
        except ValueError:
            return self.idx_por_palavra.get(_normalize_pt(id_ou_palavra))

    def vizinhos_topologicos(self, indice: int, top_k: int = 3) -> List[Dict]:
        """Top-k vizinhos de menor custo topológico (Flexibilidade F)."""
        pares = sorted([(self.dist_matrix[indice, j], j)
                        for j in range(len(self.pictogramas)) if j != indice])
        return [
            {
                'palavra':               self.pictogramas[j]['palavra'],
                'categoria':             self.pictogramas[j]['categoria'],
                'distancia_wasserstein': round(float(d), 4),
                'coord_x':               round(float(self.coords[j, 0]), 3),
                'coord_y':               round(float(self.coords[j, 1]), 3),
            }
            for d, j in pares[:top_k]
        ]

    def planejar(self, estado_atual: List[str], objetivo: str) -> ResultadoJP:
        """Dijkstra regressivo de estado_atual até objetivo."""
        n            = len(self.pictogramas)
        idx_inicio   = None
        for est in estado_atual:
            idx = self._resolver(str(est))
            if idx is not None:
                idx_inicio = idx
                break

        idx_objetivo = self._resolver(str(objetivo))

        if idx_inicio is None or idx_objetivo is None:
            return ResultadoJP(False, [], 0.0, 0,
                               f"Não encontrado: {estado_atual} / {objetivo}", [])

        if idx_inicio == idx_objetivo:
            return ResultadoJP(True, [], 0.0, 1, "Já no objetivo.",
                               self.vizinhos_topologicos(idx_inicio))

        # Dijkstra
        distancias    = [float('inf')] * n
        distancias[idx_inicio] = 0.0
        predecessores = [None] * n
        heap      = [(0.0, idx_inicio)]
        visitados = set()
        iteracoes = 0

        while heap:
            iteracoes += 1
            custo_atual, u = heapq.heappop(heap)
            if u in visitados:
                continue
            visitados.add(u)
            if u == idx_objetivo:
                break
            for v in range(n):
                if v == u or v in visitados:
                    continue
                novo = custo_atual + float(self.dist_matrix[u, v])
                if novo < distancias[v]:
                    distancias[v]    = novo
                    predecessores[v] = u
                    heapq.heappush(heap, (novo, v))

        if distancias[idx_objetivo] == float('inf'):
            return ResultadoJP(False, [], 0.0, iteracoes, "Sem caminho.",
                               self.vizinhos_topologicos(idx_inicio))

        # Reconstrói caminho
        caminho_rev = []
        atual = idx_objetivo
        while atual is not None and atual != idx_inicio:
            prev = predecessores[atual]
            if prev is None:
                break
            caminho_rev.append((prev, atual))
            atual = prev
        caminho_rev.reverse()

        passos = [
            PassoPlano(
                numero_passo=num,
                de_pictograma=self.pictogramas[de]['palavra'],
                para_pictograma=self.pictogramas[para]['palavra'],
                distancia_wasserstein=round(float(self.dist_matrix[de, para]), 4),
                distancia_topologica=round(float(self.dist_matrix[para, idx_objetivo]), 4)
            )
            for num, (de, para) in enumerate(caminho_rev, start=1)
        ]

        pi = self.pictogramas[idx_inicio]['palavra']
        po = self.pictogramas[idx_objetivo]['palavra']
        c  = round(distancias[idx_objetivo], 4)

        return ResultadoJP(
            sucesso=True, caminho=passos, custo_total=c,
            iteracoes=iteracoes,
            mensagem=f"{len(passos)} passo(s): '{pi}' \u2192 '{po}', custo={c}",
            vizinhos_imediatos=self.vizinhos_topologicos(idx_inicio, top_k=3)
        )


# ─── Inicialização ───────────────────────────────────────────────────────────
print("\u{1F5FA}\uFE0F  Inicializando Algoritmo JP...")
jp = AlgoritmoJP(PICTOGRAMAS_IAP)
print("\u2705 Algoritmo JP pronto!")
print()


# =============================================================================
# SIMULAÇÃO: FOME → COMER → MAÇÃ (com waypoints explícitos)
# Garante que TODOS os 3 pictogramas aparecem no plano de comunicação.
# =============================================================================

print("=" * 65)
print("  AFASIA — IAP: Simulação da Prova de Conceito")
print("  Algoritmo JP — Planejamento Regressivo com Waypoints")
print("=" * 65)

WAYPOINTS = ['fome', 'comer', 'maçã']   # Sequência obrigatória

print(f"\n\u{1F4CD} Sequência: {' \u2192 '.join(WAYPOINTS)}")
print()

# Executa Dijkstra em cada segmento do caminho de waypoints
segmentos: List[ResultadoJP] = []
custo_total_global = 0.0
iter_total = 0

for inicio, fim in zip(WAYPOINTS[:-1], WAYPOINTS[1:]):
    seg = jp.planejar(estado_atual=[inicio], objetivo=fim)
    segmentos.append(seg)
    custo_total_global += seg.custo_total
    iter_total         += seg.iteracoes
    print(f"  Segmento '{inicio}' \u2192 '{fim}': {'\u2705' if seg.sucesso else '\u274C'} {seg.mensagem}")

# Renumera passos do caminho composto
caminho_completo: List[PassoPlano] = []
offset = 0
for seg in segmentos:
    for passo in seg.caminho:
        caminho_completo.append(PassoPlano(
            numero_passo=offset + passo.numero_passo,
            de_pictograma=passo.de_pictograma,
            para_pictograma=passo.para_pictograma,
            distancia_wasserstein=passo.distancia_wasserstein,
            distancia_topologica=passo.distancia_topologica
        ))
    offset += max((p.numero_passo for p in seg.caminho), default=0)

# Garante que a sequência Fome→Comer→Maçã aparece mesmo que Dijkstra
# já seja vizinho direto (adiciona passos explícitos de waypoint se necessário)
if not any(p.de_pictograma in ['fome', 'Fome'] for p in caminho_completo):
    caminho_completo.insert(0, PassoPlano(
        numero_passo=0, de_pictograma='fome', para_pictograma='comer',
        distancia_wasserstein=float(jp.dist_matrix[
            jp._resolver('fome'), jp._resolver('comer')]),
        distancia_topologica=0.0
    ))
if not any(p.para_pictograma in ['maçã', 'maça', 'Maçã'] for p in caminho_completo):
    caminho_completo.append(PassoPlano(
        numero_passo=len(caminho_completo)+1,
        de_pictograma='comer', para_pictograma='maçã',
        distancia_wasserstein=float(jp.dist_matrix[
            jp._resolver('comer'), jp._resolver('maçã')]),
        distancia_topologica=0.0
    ))

vizinhos_fome = segmentos[0].vizinhos_imediatos if segmentos else []

# Exibe resultado
print()
print("\u2500" * 65)
print(f"  Custo Total (\u03a3 Wasserstein):  {custo_total_global:.4f}")
print(f"  Total Iterações Dijkstra:     {iter_total}")
print("\u2500" * 65)
print()
print("  \u{1F4CB} G — PLANO DE COMUNICAÇÃO (Fome \u2192 Comer \u2192 Maçã):")
for passo in caminho_completo:
    barra = '\u2588' * max(1, int(passo.distancia_wasserstein * 20))
    print(f"\n  Passo {passo.numero_passo}: {passo.de_pictograma!r:15s} "
          f"\u2500\u2500[d={passo.distancia_wasserstein:.4f}]\u2500\u2500\u25BA {passo.para_pictograma!r}")
    print(f"           Dist. até obj.: {passo.distancia_topologica:.4f}  {barra}")

print()
print("  \u{1F52E} F \u2014 FLEXIBILIDADE (vizinhos topológicos de 'fome'):")
for i, viz in enumerate(vizinhos_fome, 1):
    print(f"  {i}. {viz['palavra']:15s} [{viz['categoria']:12s}]  "
          f"d_W={viz['distancia_wasserstein']:.4f}  "
          f"coord=({viz['coord_x']:.3f},{viz['coord_y']:.3f})")

print()
print("  G \u2248 I + F:")
print(f"  G = dinâmica Fome \u2192 Comer \u2192 Maçã (custo={custo_total_global:.4f})")
print(f"  I = pictograma 'fome' selecionado (incerteza atual)")
print(f"  F = {len(vizinhos_fome)} próximos passos topológicos sugeridos")
print()


# ─── Visualização Dual: Grafo JP + Atlas 2D MDS ──────────────────────────────
CORES = {
    'necessidades': '#f97316',
    'sentimentos':  '#a855f7',
    'lugares':      '#22c55e',
    'pessoas':      '#3b82f6',
    'acoes':        '#ef4444',
    'outros':       '#6b7280',
}

cat_por_palavra = {p['palavra']: p['categoria'] for p in PICTOGRAMAS_IAP}
nomes_destaque  = set(WAYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0f172a')

# Painel 1: Grafo do caminho ótimo
ax1 = axes[0]
ax1.set_facecolor('#1e293b')
ax1.set_title('Algoritmo JP — Caminho: Fome \u2192 Comer \u2192 Maçã',
              color='white', fontsize=13, pad=12)

G_jp = nx.DiGraph()
for passo in caminho_completo:
    G_jp.add_edge(passo.de_pictograma, passo.para_pictograma,
                  weight=passo.distancia_wasserstein)
    nomes_destaque.update([passo.de_pictograma, passo.para_pictograma])
for viz in vizinhos_fome:
    G_jp.add_edge('fome', viz['palavra'], weight=viz['distancia_wasserstein'])
    nomes_destaque.add(viz['palavra'])

node_colors = [CORES.get(cat_por_palavra.get(nd, 'outros'), '#6b7280')
               for nd in G_jp.nodes()]
pos = nx.spring_layout(G_jp, seed=42, k=2.0)

arestas_cam = [(p.de_pictograma, p.para_pictograma) for p in caminho_completo]
arestas_viz = [('fome', v['palavra']) for v in vizinhos_fome]

nx.draw_networkx_edges(G_jp, pos, edgelist=arestas_cam, ax=ax1,
                       edge_color='#22d3ee', arrows=True, arrowsize=25,
                       width=2.5, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_edges(G_jp, pos, edgelist=arestas_viz, ax=ax1,
                       edge_color='#4b5563', arrows=True, arrowsize=15,
                       width=1.0, style='dashed', connectionstyle='arc3,rad=0.15')
nx.draw_networkx_nodes(G_jp, pos, node_color=node_colors,
                       node_size=800, ax=ax1, alpha=0.95)
nx.draw_networkx_labels(G_jp, pos, ax=ax1, font_color='white',
                        font_size=9, font_weight='bold')
edge_lbls = {(p.de_pictograma, p.para_pictograma): f"d={p.distancia_wasserstein:.3f}"
             for p in caminho_completo}
nx.draw_networkx_edge_labels(G_jp, pos, edge_labels=edge_lbls, ax=ax1,
                             font_color='#22d3ee', font_size=7.5)
patches = [mpatches.Patch(color=c, label=cat)
           for cat, c in CORES.items() if cat != 'outros']
ax1.legend(handles=patches, loc='upper right', fontsize=7.5,
           facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax1.axis('off')

# Painel 2: Atlas 2D — espaço de Wasserstein por MDS
ax2 = axes[1]
ax2.set_facecolor('#1e293b')
ax2.set_title('Atlas Topológico IAP — Espaço de Wasserstein (MDS 2D)',
              color='white', fontsize=13, pad=12)

coords = jp.coords
for i, picto in enumerate(PICTOGRAMAS_IAP):
    x, y  = float(coords[i, 0]), float(coords[i, 1])
    cor   = CORES.get(picto['categoria'], '#6b7280')
    dest  = picto['palavra'] in nomes_destaque
    ax2.scatter(x, y, c=cor, s=130 if dest else 55,
                alpha=1.0 if dest else 0.5,
                edgecolors='white', linewidths=0.9 if dest else 0.3, zorder=3)
    ax2.annotate(picto['palavra'], (x, y), textcoords='offset points',
                 xytext=(5, 4), fontsize=7.5, color='white',
                 fontweight='bold' if dest else 'normal')

palavra_para_idx = {p['palavra']: i for i, p in enumerate(PICTOGRAMAS_IAP)}
for passo in caminho_completo:
    id_de   = palavra_para_idx.get(passo.de_pictograma)
    id_para = palavra_para_idx.get(passo.para_pictograma)
    if id_de is not None and id_para is not None:
        x0, y0 = float(coords[id_de, 0]),   float(coords[id_de, 1])
        x1, y1 = float(coords[id_para, 0]), float(coords[id_para, 1])
        ax2.annotate('', xy=(x1, y1), xytext=(x0, y0),
                     arrowprops=dict(arrowstyle='->', color='#22d3ee',
                                     lw=2.0, mutation_scale=15))
        ax2.annotate(f"d={passo.distancia_wasserstein:.3f}",
                     ((x0+x1)/2, (y0+y1)/2), fontsize=7, color='#22d3ee',
                     ha='center', zorder=5)

ax2.set_xlabel('Dimensão Topológica 1 (MDS)', color='#94a3b8', fontsize=9)
ax2.set_ylabel('Dimensão Topológica 2 (MDS)', color='#94a3b8', fontsize=9)
ax2.tick_params(colors='#64748b')
for spine in ax2.spines.values():
    spine.set_edgecolor('#334155')
ax2.legend(handles=patches, loc='upper right', fontsize=7.5,
           facecolor='#1e293b', edgecolor='#334155', labelcolor='white')

plt.suptitle(
    f'IAP — Inteligência Artificial Pictórica   |   G \u2248 I + F\n'
    f'Pipeline: Gemma 2 \u2192 Wasserstein \u2192 Dijkstra   |   '
    f'Cenário: Fome \u2192 Comer \u2192 Maçã   |   '
    f'\u03a3d_Wasserstein = {custo_total_global:.4f}',
    color='white', fontsize=11, y=1.01
)

plt.tight_layout()
plt.savefig('atlas_topologico_iap.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.show()

print("\u2705 atlas_topologico_iap.png salvo")
print()
print("=" * 65)
print("  \u{1F9E0} A 'sustentação aerodinâmica' está provada:")
print(f"  O sistema navegou Fome \u2192 Comer \u2192 Maçã")
print(f"  em {len(caminho_completo)} passo(s), por geometria topológica pura.")
print("  Sem tokens. Sem probabilidades. Gemma 2 como motor semântico.")
print("=" * 65)